# Notebook 02: HuggingFace Trainer API

Same experiments using HuggingFace Trainer and AutoModelForSequenceClassification. For LoRA we use the PEFT library (not lora_model.py).

## Section 1 — Setup

Same as Notebook 01: imports, seed=42, device, CONFIG, **Model Selection Justification**.

### MODEL SELECTION JUSTIFICATION

We use **XLM-RoBERTa-base** and **MARBERTv2** (not BLOOM/mT5/AceGPT/Jais): XLM-RoBERTa for cross-lingual code-switching; MARBERTv2 for Arabic dialect strength. Documented in Section 4 (Methodology).

In [ ]:
import os
import random
import sys
from pathlib import Path
import numpy as np
import torch
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config_path = ROOT / "configs" / "config.yaml"
CONFIG = yaml.safe_load(config_path.read_text()) if config_path.exists() else {}
print("Device:", device)

## Section 2 — Data Loading and Exploration

Same as Notebook 01.

In [ ]:
master_path = ROOT / "data" / "processed" / "master_dataset.csv"
if not master_path.exists():
    master_path = ROOT / "preprocessing" / "data" / "processed" / "training_dataset.csv"
df = pd.read_csv(master_path)
print(df.shape, df["label"].value_counts().sort_index())

## Sections 3–4 — Augmentation and Tokenizer

Same as Notebook 01 (optional runs).

## Section 5–8 — Experiments with HuggingFace Trainer

**Key difference**: We use `AutoModelForSequenceClassification` and `TrainingArguments` + `Trainer` (not PyTorchSentimentTrainer). For LoRA we use **LoraConfig from the PEFT library** (not lora_model.py).

> This uses the HuggingFace PEFT library for LoRA. Notebook 01 implements LoRA from scratch in lora_layer.py to demonstrate the mathematical principles. Both approaches produce equivalent results — verified in Section 9.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import datasets

max_len = CONFIG.get("data", {}).get("max_sequence_length", 128)
batch_size = CONFIG.get("training", {}).get("batch_size", 16)

def prepare_dataset(path, tokenizer):
    df = pd.read_csv(path)
    if "split" not in df.columns:
        df["split"] = "train"
    train = df[df["split"] == "train"]
    val = df[df["split"] == "val"]
    test = df[df["split"] == "test"]
    def tokenize(ex):
        return tokenizer(ex["text"], truncation=True, max_length=max_len, padding="max_length")
    for part, name in [(train, "train"), (val, "val"), (test, "test")]:
        if len(part) == 0:
            continue
        d = datasets.Dataset.from_pandas(part[["text", "label"]].reset_index(drop=True))
        d = d.map(tokenize, batched=True, remove_columns=["text"])
        d.set_format("torch")
    train_d = datasets.Dataset.from_pandas(train[["text", "label"]].reset_index(drop=True))
    train_d = train_d.map(lambda ex: tokenizer(ex["text"], truncation=True, max_length=max_len, padding="max_length"), batched=True, remove_columns=["text"])
    val_d = datasets.Dataset.from_pandas(val[["text", "label"]].reset_index(drop=True)) if len(val) > 0 else None
    if val_d is not None:
        val_d = val_d.map(lambda ex: tokenizer(ex["text"], truncation=True, max_length=max_len, padding="max_length"), batched=True, remove_columns=["text"])
    return train_d, val_d

xlm_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
train_d, val_d = prepare_dataset(master_path, xlm_tok)
print("Train size:", len(train_d))

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}

model_hf = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)
args = TrainingArguments(
    output_dir=str(ROOT / "outputs" / "hf_xlm_full"),
    num_train_epochs=CONFIG.get("training", {}).get("max_epochs", 10),
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=CONFIG.get("training", {}).get("full_finetuning", {}).get("learning_rate", 2e-5),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
)
trainer_hf = Trainer(model=model_hf, args=args, train_dataset=train_d, eval_dataset=val_d, compute_metrics=compute_metrics)
trainer_hf.train()
print(trainer_hf.evaluate())

In [ ]:
# LoRA with PEFT
model_lora = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)
lora_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["query", "key", "value"])
model_lora = get_peft_model(model_lora, lora_config)
args_lora = TrainingArguments(output_dir=str(ROOT / "outputs" / "hf_xlm_lora"), num_train_epochs=10, per_device_train_batch_size=batch_size, learning_rate=3e-4, evaluation_strategy="epoch")
trainer_lora = Trainer(model=model_lora, args=args_lora, train_dataset=train_d, eval_dataset=val_d, compute_metrics=compute_metrics)
trainer_lora.train()

## Section 9 — Cross-Framework Comparison

Load results from both notebooks; side-by-side PyTorch vs HuggingFace Trainer; training time, memory, macro-F1, BLEU; code complexity (lines per experiment).

In [ ]:
# Load PyTorch results from 01 (if saved) and compare with HF Trainer results
from evaluation.standard_metrics import SentimentEvaluator
eval_hf = SentimentEvaluator()
print("Compare: PyTorch (Notebook 01) vs HuggingFace Trainer (this notebook).")
print("Metrics and code complexity can be compared in the report.")